# Phase 0. 데이터 통합

**목적**: 60개 월별 CSV 파일을 하나로 통합하고 파생 컬럼을 추가한다. 값 제거는 하지 않는다.

**입력 데이터**: `2021/~2025/anonymized_YYYYMM.csv.gz` (60개 파일)

**출력 데이터**: `data/processed/all_data.parquet` (16,868,714행, 원본 그대로)

**방침**: 음수값, 극단값, NaN 행 제거는 EDA(Phase 1) 이후 종별별 분포를 확인한 뒤 Phase 2에서 처리한다.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import holidays
import warnings
warnings.filterwarnings('ignore')

HOUR_COLS = [f'{i}시' for i in range(1, 25)]

## 0-1. 데이터 통합

60개 월별 CSV.gz 파일을 하나의 DataFrame으로 병합한다.

In [ ]:
all_dfs = []
for year in range(2021, 2026):
    files = sorted(glob.glob(f'../{ year}/anonymized_{year}*.csv.gz'))
    for f in files:
        df = pd.read_csv(f, encoding='utf-8-sig')
        all_dfs.append(df)

df = pd.concat(all_dfs, ignore_index=True)
print(f'통합 완료: {len(df):,} rows, {df.shape[1]} cols')

## 컬럼 정리 및 파생 컬럼 추가

- `총` → `총사용량` rename
- `유형` 컬럼 drop (전 기간 `-`만 존재)
- 날짜 파생: 연도, 월, 일, 요일, 계절, 난방시즌, 공휴일

In [ ]:
# 컬럼명 정리
df.rename(columns={'총': '총사용량'}, inplace=True)
df.drop(columns=['유형'], inplace=True)

# 날짜 변환 및 파생 컬럼
df['날짜'] = pd.to_datetime(df['날짜'])
df['연도'] = df['날짜'].dt.year
df['월'] = df['날짜'].dt.month
df['일'] = df['날짜'].dt.day
df['요일'] = df['날짜'].dt.dayofweek  # 0=월 ~ 6=일

season_map = {12:'겨울', 1:'겨울', 2:'겨울', 3:'봄', 4:'봄', 5:'봄',
              6:'여름', 7:'여름', 8:'여름', 9:'가을', 10:'가을', 11:'가을'}
df['계절'] = df['월'].map(season_map)
df['난방시즌'] = df['월'].isin([11, 12, 1, 2, 3])

# 공휴일
kr_holidays = holidays.KR(years=range(2021, 2026))
df['공휴일'] = df['날짜'].dt.date.map(lambda d: d in kr_holidays)

print(f'컬럼: {list(df.columns)}')
print(f'shape: {df.shape}')

## 0-2. 총사용량 재계산 및 무결성 검증

In [ ]:
# 총사용량 재계산
df['총사용량'] = df[HOUR_COLS].sum(axis=1)

# 연도별 설비 수
print('연도별 설치(열량계) 수:')
print(df.groupby('연도')['설치'].nunique().to_string())

# 종별 변경 이력
type_per_inst = df.groupby('설치')['종별'].nunique()
changed = type_per_inst[type_per_inst > 1]
print(f'\n종별 변경 설비 수: {len(changed)}')

# 현황 참고 (제거하지 않음)
neg_count = (df[HOUR_COLS] < 0).sum().sum()
over100 = (df[HOUR_COLS] > 100).sum().sum()
nan_count = df[HOUR_COLS].isnull().sum().sum()
print(f'\n[EDA에서 확인 예정]')
print(f'  음수값: {neg_count:,}건')
print(f'  >100 Gcal: {over100:,}건')
print(f'  NaN: {nan_count:,}건')

print(f'\n총 설비 수: {df["설치"].nunique():,}')
print(f'총 행 수: {len(df):,}')
print(f'날짜 범위: {df["날짜"].min().date()} ~ {df["날짜"].max().date()}')

## 0-5. 저장

In [ ]:
df.to_parquet('../data/processed/all_data.parquet', index=False)
print(f'저장 완료: data/processed/all_data.parquet')
print(f'파일 크기: {os.path.getsize("../data/processed/all_data.parquet") / 1e6:.1f} MB')

## 핵심 발견사항

1. **16,868,714행 통합 완료** — 값 제거 없이 원본 그대로 보존
2. **종별 변경 설비 0건** — 전 기간 매핑 일관성 확인
3. **설비 수 8,895(2021) → 9,883(2025)** — 5년간 약 11% 증가
4. **유형 컬럼은 전 기간 `-`만 존재** — drop 처리
5. **EDA 후 처리 예정**: 음수값 7,881건, 극단값(>100Gcal) 3,482건, NaN ~104,000행